In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
df=pd.read_csv('E:\\Projects\\social_impact\\Social_media_impact_on_life.csv')

In [4]:
df.head(1)

,Student_ID,Age,Gender,Academic_Level,Primary_Platform,Daily_Usage_Hours,Weekend_Extra_Hours,Device_Type,Sleep_Duration_Hours,Sleep_Quality_Score,Late_Night_Usage,Social_Comparison_Frequency,Perceived_Stress_Score,Mental_Health_Index,Academic_Performance_GPA,Overall_Impact
0,STU_2024000,19,Female,Undergraduate,Snapchat,7.5,1.5,Smartphone,5.9,2,True,Always,20.0,73,2.73,Neutral


In [5]:
df=df.drop(columns='Student_ID')

In [12]:
df.sample()

,Age,Gender,Academic_Level,Primary_Platform,Daily_Usage_Hours,Weekend_Extra_Hours,Device_Type,Sleep_Duration_Hours,Sleep_Quality_Score,Late_Night_Usage,Social_Comparison_Frequency,Perceived_Stress_Score,Mental_Health_Index,Academic_Performance_GPA,Overall_Impact
1959,19,Male,Undergraduate,Instagram,2.4,1.6,Smartphone,7.9,4,False,Always,12.0,95,3.37,Beneficial


In [9]:
X=df.drop(columns='Overall_Impact')
y=df['Overall_Impact']

In [11]:
y.value_counts()

Overall_Impact
Beneficial    3681
Neutral        654
Negative       165
Name: count, dtype: int64

In [ ]:
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 4500 entries, 0 to 4499
Data columns (total 14 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Age                          4500 non-null   int64  
 1   Gender                       4500 non-null   str    
 2   Academic_Level               4500 non-null   str    
 3   Primary_Platform             4500 non-null   str    
 4   Daily_Usage_Hours            4500 non-null   float64
 5   Weekend_Extra_Hours          4500 non-null   float64
 6   Device_Type                  4500 non-null   str    
 7   Sleep_Duration_Hours         4500 non-null   float64
 8   Sleep_Quality_Score          4500 non-null   int64  
 9   Late_Night_Usage             4500 non-null   bool   
 10  Social_Comparison_Frequency  4500 non-null   str    
 11  Perceived_Stress_Score       4454 non-null   float64
 12  Mental_Health_Index          4500 non-null   int64  
 13  Academic_Performance_GPA     

In [ ]:
print(f"\n{(df.select_dtypes(exclude=['bool','str']).columns.tolist())} ")




['Age', 'Daily_Usage_Hours', 'Weekend_Extra_Hours', 'Sleep_Duration_Hours', 'Sleep_Quality_Score', 'Perceived_Stress_Score', 'Mental_Health_Index', 'Academic_Performance_GPA'] 


In [ ]:
print(f'numerical columns \n {len(df.select_dtypes(exclude=['bool','str']).columns.tolist())}')

numerical columns 
 8


In [ ]:
print(f"Categorical columns {df.select_dtypes(include=['bool','str']).columns.tolist()}")

Categorical columns ['Gender', 'Academic_Level', 'Primary_Platform', 'Device_Type', 'Late_Night_Usage', 'Social_Comparison_Frequency', 'Overall_Impact']


In [ ]:
df.isnull().sum()

Age                             0
Gender                          0
Academic_Level                  0
Primary_Platform                0
Daily_Usage_Hours               0
Weekend_Extra_Hours             0
Device_Type                     0
Sleep_Duration_Hours            0
Sleep_Quality_Score             0
Late_Night_Usage                0
Social_Comparison_Frequency     0
Perceived_Stress_Score         46
Mental_Health_Index             0
Academic_Performance_GPA       85
Overall_Impact                  0
dtype: int64

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y,
    test_size=0.2, 
    random_state=42,
    stratify=y)

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(y_train)
y_test = label_encoder.transform(y_test)

In [ ]:
skew_numerical=['Age','Daily_Usage_Hours','Weekend_Extra_Hours','Sleep_Duration_Hours','Sleep_Quality_Score','Perceived_Stress_Score','Mental_Health_Index','Academic_Performance_GPA']
label_categorical=['Academic_Level','Social_Comparison_Frequency']
ohe_categorical=[col for col in X.select_dtypes(include=['bool','str']).columns.tolist() if col not in label_categorical]
ohe_categorical

['Gender', 'Primary_Platform', 'Device_Type', 'Late_Night_Usage']

In [ ]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder,FunctionTransformer,OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [ ]:

skew_num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(
        np.log1p,
        feature_names_out='one-to-one'
    )),
    ('scaler', StandardScaler())
])

ohe_pipeline = Pipeline(steps=[
    ('one_hot_encoder', OneHotEncoder(
        handle_unknown='ignore'
    ))
])

ordinal_pipeline = Pipeline(steps=[
    ('ordinal_encoder', OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

In [ ]:
preprocessor = ColumnTransformer([
    ('num', skew_num_pipeline, skew_numerical),
    ('cat_label', ordinal_pipeline, label_categorical),
    ('cat_ohe', ohe_pipeline, ohe_categorical)
])

In [ ]:
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat_label', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``fea

In [ ]:
from imblearn.over_sampling import SMOTENC

categorical_columns = label_categorical + ohe_categorical
categorical_indices = [
    X_train.columns.get_loc(column)
    for column in categorical_columns
]

smote = SMOTENC(
    categorical_features=categorical_indices,
    random_state=42
)
print(categorical_indices)

[2, 10, 1, 3, 6, 9]


In [ ]:
X_train_smote=preprocessor.fit_transform(X_train)

In [ ]:
categorical_indices = list(
    range(
        len(skew_numerical),
        len(skew_numerical) + len(label_categorical) + len(ohe_categorical)
    )
)

print(categorical_indices)

[8, 9, 10, 11, 12, 13]


In [ ]:
smote = SMOTENC(
    categorical_features=categorical_indices,
    random_state=42
)

X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train_smote,
    y_train
)

In [ ]:
print("Before SMOTENC:")
print("X_train:", X_train.shape)
print("Before SMOTENC:")
print("X_train:", X_train.shape)
print(pd.Series(y_train).value_counts())

print("\nAfter SMOTENC:")
print("X_train_resampled:", X_train_resampled.shape)
print(pd.Series(y_train_resampled).value_counts())

print("\nAfter SMOTENC:")
print("X_train_resampled:", X_train_resampled.shape)
print(pd.Series(y_train_resampled).value_counts())

Before SMOTENC:
X_train: (3600, 14)
Before SMOTENC:
X_train: (3600, 14)
0    2945
2     523
1     132
Name: count, dtype: int64

After SMOTENC:
X_train_resampled: (8835, 26)
0    2945
2    2945
1    2945
Name: count, dtype: int64

After SMOTENC:
X_train_resampled: (8835, 26)
0    2945
2    2945
1    2945
Name: count, dtype: int64


In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr_model.fit(
    X_train_resampled,
    y_train_resampled
)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in 

In [ ]:
# Test preprocessing — SAME preprocessor_smote
X_test_processed = preprocessor.transform(X_test)

# Prediction
lr_preds = lr_model.predict(X_test_processed)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_test, lr_preds))

print("Precision:", precision_score(
    y_test,
    lr_preds,
    average='weighted'
))

print("Recall:", recall_score(
    y_test,
    lr_preds,
    average='weighted'
))

print("F1 Score:", f1_score(
    y_test,
    lr_preds,
    average='weighted'
))

print("\nClassification Report:")
print(classification_report(
    y_test,
    lr_preds,
    target_names=label_encoder.classes_
))

print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test,
    lr_preds
))

Accuracy: 0.9755555555555555
Precision: 0.9767915076806915
Recall: 0.9755555555555555
F1 Score: 0.975958864073224

Classification Report:
              precision    recall  f1-score   support

  Beneficial       1.00      0.98      0.99       736
    Negative       0.86      0.97      0.91        33
     Neutral       0.90      0.94      0.92       131

    accuracy                           0.98       900
   macro avg       0.92      0.96      0.94       900
weighted avg       0.98      0.98      0.98       900


Confusion Matrix:
[[723   0  13]
 [  0  32   1]
 [  3   5 123]]


In [ ]:
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'liblinear','newton-cg','newton-cholesky','sag','saga'],
}

In [ ]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    estimator=LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(
    X_train_resampled,
    y_train_resampled
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits


e:\Projects\social_impact\venv\Lib\site-packages\sklearn\model_selection\_validation.py:493: FitFailedWarning: 
25 fits failed out of a total of 150.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
25 fits failed with the following error:
Traceback (most recent call last):
  File "e:\Projects\social_impact\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 855, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\Projects\social_impact\venv\Lib\site-packages\sklearn\base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "e:\Projects\social_impact\venv\Lib\site-packages\sklearn\linear_model\_logistic.py", line 14

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegre...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.01, 0.1, ...], 'solver': ['lbfgs', 'liblinear', ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see ho

In [ ]:
best_params=grid_search.best_params_
best_params

{'C': 100, 'solver': 'newton-cholesky'}

In [ ]:
tuned_lr=grid_search.best_estimator_

In [ ]:
from sklearn.metrics import accuracy_score

tuned_preds = tuned_lr.predict(X_test_processed)

accuracy = accuracy_score(y_test, tuned_preds)

print("Tuned Logistic Regression Accuracy:", accuracy)
print("Tuned Logistic Regression Accuracy (%):", accuracy * 100)

Tuned Logistic Regression Accuracy: 0.9788888888888889
Tuned Logistic Regression Accuracy (%): 97.88888888888889


In [ ]:
print("Baseline Accuracy:", accuracy_score(y_test, lr_preds))
print("Tuned Accuracy:", accuracy_score(y_test, tuned_preds))

Baseline Accuracy: 0.9755555555555555
Tuned Accuracy: 0.9788888888888889


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# 1. Random Forest parameter grid
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}


# 2. Base Random Forest model
rf_model = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)


# 3. GridSearchCV
grid_search_rf = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid_rf,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1,
    verbose=1
)


# 4. Train GridSearch
grid_search_rf.fit(
    X_train_resampled,
    y_train_resampled
)


# 5. Get the best Random Forest model
tuned_rf = grid_search_rf.best_estimator_


# 6. Best parameters
print("Best Parameters:")
print(grid_search_rf.best_params_)

print("\nBest CV Macro F1:")
print(grid_search_rf.best_score_)


# 7. Prediction on test set
rf_preds = tuned_rf.predict(X_test_processed)


# 8. Accuracy
rf_accuracy = accuracy_score(
    y_test,
    rf_preds
)

print("\nAccuracy:", rf_accuracy)
print("Accuracy (%):", rf_accuracy * 100)


# 9. Other metrics
print("\nPrecision:", precision_score(
    y_test,
    rf_preds,
    average='weighted'
))

print("Recall:", recall_score(
    y_test,
    rf_preds,
    average='weighted'
))

print("Macro F1:", f1_score(
    y_test,
    rf_preds,
    average='macro'
))

print("Weighted F1:", f1_score(
    y_test,
    rf_preds,
    average='weighted'
))


# 10. Classification Report
print("\nClassification Report:")
print(classification_report(
    y_test,
    rf_preds,
    target_names=label_encoder.classes_
))


# 11. Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test,
    rf_preds
))

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


KeyboardInterrupt: 

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Parameter grid
param_grid_rf = {
    'n_estimators': [150, 300, 450],
    'max_depth': [5, 15, 25, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

# Base Random Forest
rf_model = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

# Grid Search
grid_search_rf = GridSearchCV(
    estimator=rf_model,
    param_grid=param_grid_rf,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1,
    verbose=1
)

# Fit
grid_search_rf.fit(
    X_train_resampled,
    y_train_resampled
)

# Best model
tuned_rf = grid_search_rf.best_estimator_

# Best parameters
print("Best Parameters:")
print(grid_search_rf.best_params_)

print("\nBest CV Macro F1:")
print(grid_search_rf.best_score_)

# Prediction
rf_preds = tuned_rf.predict(X_test_processed)

# Metrics
print("\nAccuracy:", accuracy_score(y_test, rf_preds))

print("Precision:", precision_score(
    y_test,
    rf_preds,
    average='weighted'
))

print("Recall:", recall_score(
    y_test,
    rf_preds,
    average='weighted'
))

print("Macro F1:", f1_score(
    y_test,
    rf_preds,
    average='macro'
))

print("Weighted F1:", f1_score(
    y_test,
    rf_preds,
    average='weighted'
))

# Classification report
print("\nClassification Report:")
print(classification_report(
    y_test,
    rf_preds
))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test,
    rf_preds
))

Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Best Parameters:
{'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}

Best CV Macro F1:
0.9929801239945288

Accuracy: 0.9644444444444444
Precision: 0.9649852317496839
Recall: 0.9644444444444444
Macro F1: 0.9149306820407959
Weighted F1: 0.9646735129248353

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.98       736
           1       0.86      0.91      0.88        33
           2       0.87      0.89      0.88       131

    accuracy                           0.96       900
   macro avg       0.91      0.93      0.91       900
weighted avg       0.96      0.96      0.96       900


Confusion Matrix:
[[722   0  14]
 [  0  30   3]
 [ 10   5 116]]
